In [1]:
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from torch.utils.data import Dataset
from pathlib import Path
import json
import torch
import pandas as pd
import numpy as np

C:\Users\Barderus_Legion\PycharmProjects\TrustNet\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = NOTEBOOK_DIR.parent / "data"
MODELS_DIR = NOTEBOOK_DIR.parent / "models"
ARTIFACTS_DIR = NOTEBOOK_DIR.parent / "artifacts" / "training"

FAKE_NEWS_MODEL_DIR = MODELS_DIR / "fake_news_model" / "distilbert_fakenews_model"
FAKE_NEWS_TOKENIZER_DIR = MODELS_DIR / "fake_news_model" / "distilbert_fakenews_tokenizer"
FAKE_NEWS_RUN_DIR = ARTIFACTS_DIR / "fake_news_transformer"

FAKE_NEWS_MODEL_DIR.parent.mkdir(parents=True, exist_ok=True)
FAKE_NEWS_TOKENIZER_DIR.parent.mkdir(parents=True, exist_ok=True)
FAKE_NEWS_RUN_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
import accelerate
print(accelerate.__version__)

1.13.0


In [4]:
df = pd.read_csv(DATA_DIR / "preprocessed" / "fakenews_preprocessed.csv")

In [5]:
texts = df["text"].fillna("").astype(str).tolist()
labels = df["real"].astype(int).tolist()

In [6]:
df["real"].value_counts()

real
1    35619
0    26507
Name: count, dtype: int64

In [7]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

In [8]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
max_len = 128

C:\Users\Barderus_Legion\PycharmProjects\TrustNet\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Barderus_Legion\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [9]:
train_enc = tokenizer(
    train_texts,
    padding="max_length",
    truncation=True,
    max_length=max_len
)

In [10]:
val_enc = tokenizer(
    val_texts,
    padding="max_length",
    truncation=True,
    max_length=max_len
)

In [11]:
class FakeNewsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels  # list of ints

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        # Each field in encodings is a list/array of token ids per example
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [12]:
train_dataset = FakeNewsDataset(train_enc, train_labels)
val_dataset   = FakeNewsDataset(val_enc,   val_labels)

In [13]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 9783.55it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc  = accuracy_score(labels, preds)
    prec = precision_score(labels, preds)
    rec  = recall_score(labels, preds)
    f1   = f1_score(labels, preds)

    return {
        "accuracy":  acc,
        "precision": prec,
        "recall":    rec,
        "f1":        f1
    }

In [15]:
training_args = TrainingArguments(
    output_dir=str(FAKE_NEWS_RUN_DIR / "trainer_output"),
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    logging_dir=str(FAKE_NEWS_RUN_DIR / "logs"),

    # CPU fixes
    no_cuda=True,
    fp16=False,
    bf16=False,
    torch_compile=False,
)

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

In [ ]:
metrics = trainer.evaluate()
print(metrics)

## Final artifact validation

Before saving the fake-news transformer artifacts, verify the downstream label contract and persist a small training summary.


In [ ]:
required_columns = ["text", "real"]
missing_columns = [column for column in required_columns if column not in df.columns]
assert not missing_columns, f"Missing required columns before training export: {missing_columns}"
assert set(df["real"].dropna().astype(int).unique()) <= {0, 1}, "The fake-news labels must remain binary."

model.config.id2label = {0: "FAKE", 1: "REAL"}
model.config.label2id = {"FAKE": 0, "REAL": 1}

summary = {
    "dataset_rows": int(len(df)),
    "train_size": int(len(train_texts)),
    "validation_size": int(len(val_texts)),
    "label_distribution": {str(k): int(v) for k, v in df["real"].value_counts().to_dict().items()},
    "metrics": {key: float(value) for key, value in metrics.items() if isinstance(value, (int, float))},
    "model_dir": str(FAKE_NEWS_MODEL_DIR),
    "tokenizer_dir": str(FAKE_NEWS_TOKENIZER_DIR),
}

print(json.dumps(summary, indent=2))
with (FAKE_NEWS_RUN_DIR / "metrics.json").open("w", encoding="utf-8") as file_handle:
    json.dump(summary, file_handle, indent=2)


In [ ]:
trainer.save_model(str(FAKE_NEWS_MODEL_DIR))
tokenizer.save_pretrained(str(FAKE_NEWS_TOKENIZER_DIR))


In [ ]:
text1 = "The Associated Press and reams of other media outlets reported that JD Vance said “school shootings are a ‘fact of life’.In fact, Vance said that “psychos” who “want to make headlines” are a “fact of life”—not “school shootings.” He then said, “We have got to bolster security at our schools."

In [ ]:
text2 = "CNN’s Jake Tapper reported that Donald Trump said “that as commander in chief, he will contemplate using the United States military or National Guard to go after his political opponents, including Democrats” like “Adam Schiff. In fact, Trump was answering a question about “agitators” who would sow “chaos on election day,” like the “Afghan refugee charged with plotting a U.S. election day massacre.” He was not talking about Americans who “don’t support him” but “sick people, radical-left lunatics,” who’ve rioted, committed arson, and murdered people. Furthermore, he was talking about the 2024 election while the military is not under his command."

In [ ]:
labels = df["real"].astype(int).tolist()
labels

In [ ]:
df["real"].value_counts()

In [ ]:
set(labels)

In [ ]:
LABELS = ["Fake", "Real"]

In [ ]:
def predict_text(text):
    model.eval()
    inputs = tokenizer(
        [text],
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)[0]

    pred_idx = probs.argmax().item()
    confidence = probs[pred_idx].item()

    return LABELS[pred_idx], confidence

In [ ]:
label1, conf1 = predict_text(text1)
label2, conf2 = predict_text(text2)

print("Text 1 Prediction:", label1, f"(confidence {conf1:.4f})")
print("Text 2 Prediction:", label2, f"(confidence {conf2:.4f})")


In [ ]:
df["real"].value_counts(normalize=True)

In [ ]:
df.sample(10)[["text","real"]]

In [ ]:
df.index.is_monotonic_increasing

In [ ]:
df[df["real"] == 0].head(5)

In [ ]:
df[df.duplicated("text", keep=False)].head(10)

In [ ]:
df.sample(5)[["text","real"]]